In [2]:
from embedder import Embedder

embedder = Embedder()
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(len(v))
print(v[0])

384
-0.02058203437252893


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

In [4]:
page = next(d for d in documents if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")

page_vec = embedder.encode(page["content"])
similarity = page_vec.dot(v)
similarity

np.float64(0.36107027225589694)

In [5]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [6]:
texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)

scores = X.dot(v)
best = chunks[scores.argmax()]
best["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

q_vec = embedder.encode("What metric do we use to evaluate a search engine?")
results = vindex.search(q_vec, num_results=5)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [ ]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"

vec_results = vindex.search(embedder.encode(query), num_results=5)
text_results = text_index.search(query, num_results=5)

vec_files = {r["filename"] for r in vec_results}
text_files = {r["filename"] for r in text_results}

vec_files - text_files